# 01 · Train (or load) the β-VAE prior

The decoder *is* the prior: `z ~ N(0, I)`, `x = decoder(z)`. Everything
downstream — MAP in notebook 03, NUTS in notebook 04 — uses exactly that, so the
one thing that matters here is whether **z carries information about x**.

## The number to watch: `kl`

The β-VAE loss has two terms, and it is only an ELBO if both are **sums** over
their own axes: a Gaussian log-density summed over the 16384 pixels, plus a KL
summed over the 128 latent dimensions. Averaging each instead (the
obvious-looking thing — `jnp.mean` on both) reweights the KL by
`n_pixels / latent_dim` = **128×**, so `beta=1.0` silently behaves like
`beta=128`.

In the log that looks like this:

```
epoch  10  loss 0.0615  recon 0.0615  kl 0.0000      <- posterior collapse
```

`kl → 0` means `q(z|x) = N(0, I)` for *every* x. The encoder has given up, z
says nothing about the image, and the decoder returns the same average knee
whatever you feed it. The loss keeps falling and nothing errors — but the prior
is worthless, and MAP has nothing to work with. A healthy 128-d latent here
sits in the **tens to hundreds of nats**.

`sigma_x` is the assumed per-pixel observation noise, and it sets the real
trade-off: smaller trusts the pixels and sharpens, larger trusts the prior and
smooths.

## Honest expectations at this data scale

With ~100 training slices from 3 volumes this prior **overfits** — hold-out
autoencoding runs several dB below training. That is a data limit, not a
hyperparameter you can tune away; `CHECKPOINTS.md` assumes 1–2k slices. Train
on `split='train'`, and report every number on `split='test'`.

In [1]:
import jax
from mrigen.train_vae import train, load_model

### Option A — train

About 12 minutes on a laptop CPU for 300 epochs. The checkpoint saved is the
one with the best **validation** MSE, not the last epoch.

In [2]:
LATENT_DIM = 128   # Contract 4: keep in 128-256 so NUTS still mixes
BETA = 1.0
SIGMA_X = 0.1      # per-pixel observation noise; the sharpness knob
EPOCHS = 300

model = train('../data/processed',
              latent_dim=LATENT_DIM, beta=BETA, sigma_x=SIGMA_X,
              epochs=EPOCHS, batch_size=16, lr=1e-3,
              augment_data=True,          # ~100 slices is not enough without it
              out='../checkpoints/vae_128.eqx')

training on 98 slices from 3 volume(s) (split='train'), validating on 16


  arch=VAE latent_dim=128 beta=1.0 sigma_x=0.1 epochs=300 batch=16 lr=0.001 augment=True


epoch    0  loss   224906.7  mse 0.27454  kl    0.01  val_mse 0.25541  *


epoch   10  loss    27489.4  mse 0.03312  kl  354.81  val_mse 0.02545  *


epoch   20  loss    12758.4  mse 0.01533  kl  203.13  val_mse 0.01652  *


...

epoch  270  loss     2713.2  mse 0.00299  kl  264.30  val_mse 0.00363  *


epoch  280  loss     2748.8  mse 0.00303  kl  266.59  val_mse 0.00364


epoch  290  loss     2673.5  mse 0.00294  kl  265.69  val_mse 0.00361  *


epoch  299  loss     2683.6  mse 0.00295  kl  268.78  val_mse 0.00359


best val_mse 0.00358 at epoch 297 (PSNR 24.46 dB) -> ../checkpoints/vae_128.eqx


### Option B — load a checkpoint, and check that it is healthy

Run this after either route. Three numbers decide whether the prior is usable.

In [3]:
import numpy as np
from mrigen.data import FastMRISlices
from mrigen.metrics import psnr

vae = load_model('../checkpoints/vae_128.eqx', latent_dim=128)
print('loaded VAE, latent_dim =', vae.latent_dim)

# The health check, in three numbers. Run this on any checkpoint before you
# trust a reconstruction built on it.
import jax.numpy as jnp


def health(split, n=10):
    ds = FastMRISlices('../data/processed', split=split)
    idx = (np.linspace(0.2, 0.8, n) * (len(ds) - 1)).round().astype(int)
    ps, kls = [], []
    for i in idx:
        x = jnp.asarray(ds[i])
        mu, logvar = vae.encoder(x)
        ps.append(psnr(np.asarray(x), np.asarray(vae.decoder(mu)), data_range=1.0))
        kls.append(float(-0.5 * jnp.sum(1 + logvar - mu**2 - jnp.exp(logvar))))
    return float(np.mean(ps)), float(np.mean(kls))


tr_psnr, tr_kl = health('train')
va_psnr, va_kl = health('val')
print(f"\n{'':<8}{'autoencode PSNR':>18}{'KL (nats)':>12}")
print(f"{'train':<8}{tr_psnr:>18.2f}{tr_kl:>12.1f}")
print(f"{'val':<8}{va_psnr:>18.2f}{va_kl:>12.1f}")
print(f"\ngeneralisation gap: {tr_psnr - va_psnr:+.2f} dB")
print(f"KL per latent dim:  {va_kl / vae.latent_dim:.2f} nats")

if va_kl < 1.0:
    print("\n!! KL is ~0: the posterior has collapsed and this prior is unusable.")
elif tr_psnr - va_psnr > 2.0:
    print("\n   A large gap means overfitting — expected at this data scale. The "
          "held-out\n   number is the honest one; more volumes is the only real fix.")
else:
    print("\n   Healthy: z carries information and the gap is small.")

loaded VAE, latent_dim = 128



           autoencode PSNR   KL (nats)
train                24.80       253.0
val                  21.08       250.9

generalisation gap: +3.72 dB
KL per latent dim:  1.96 nats

   A large gap means overfitting — expected at this data scale. The held-out
   number is the honest one; more volumes is the only real fix.
